In [1]:
#packages
import numpy as np
import tensorflow as tf
from tensorflow import keras
from recsys_utils import *

In [4]:
#load data
X, W, b, num_movies, num_features, num_users = load_precalc_params_small()
Y, R = load_ratings_small()

print('Y', Y.shape, 'R', R.shape)
print('X', X.shape)
print('W', W.shape)
print('b', b.shape)
print('num_feature', num_features)
print('num_movies', num_movies)
print('num_users', num_users)

Y (4778, 443) R (4778, 443)
X (4778, 10)
W (443, 10)
b (1, 443)
num_feature 10
num_movies 4778
num_users 443


In [5]:
#compute matrix statistics like average rating
tsmean = np.mean(Y[0, R[0, :].astype(bool)])
print(f'Average rating for movies 1 :  {tsmean:0.3f} / 5')

Average rating for movies 1 :  3.400 / 5


In [56]:
#collaborative filtering learning algorithm - minimizes the squared error
#colab filtering cost function
def cofi_cost_func(X, W, b, Y, R, lambda_):
    ''' 
    Returns the cost for the content-based filtering
    
    Args:
        X (ndarray (num_movies, num_features)) : matrix of item features
        W (ndarray(num_users, num_features))   : matrix of user parameters
        b (ndarray (1, num_users))             : vector of user parameters
        Y (ndarray (num_movies, num_users)) : matrix of user ratings of movies
        R (ndarray (num_movies, num_users)) : matrix, where R(i,j)=1 ifthe i-th
         movie was rated by the j-th user
        lambda_ (float) : regularization parameter
    
    Returns:
        J (float) : Cost

        '''
    nm, nu = Y.shape
    J = 0
    #for the users list
    for j in range(nu):
        w = W[j,:]
        b_j = b[0,j]
        #for the user-rating list
        for i in range(nm):
            x = X[i,:]
            y = Y[i,j]
            r = R[i,j]
            J += np.square(r * (np.dot(w,x) + b_j - y))# without regularization
            
    J = J/2
    J += (lambda_/2) * (np.sum(np.square(W)) + np.sum(np.square(X))) #with regularization
    return J


In [57]:
#check the above implementation
#reduce the data set size so that this runs faster
num_users_r = 4
num_movies_r = 5
num_features_r = 3

X_r = X[:num_movies_r, :num_features_r]
W_r = W[:num_users_r, :num_features_r]
b_r = b[0, :num_users_r].reshape(1,-1)
Y_r = Y[:num_movies_r, :num_users_r]
R_r = R[:num_movies_r, :num_users_r]

#evaluate cost function  without regularization
J = cofi_cost_func(X_r, W_r, b_r, Y_r, R_r, 0);
print(f'Cost: {J:0.2f}')

Cost: 13.67


In [58]:
#evaluate cost function with regularization
J = cofi_cost_func(X_r, W_r, b_r, Y_r, R_r, 0.15);
print(f'Cost (with regularization): {J:0.2f}')

Cost (with regularization): 15.11


In [53]:
#unit test
from public_tests import *
test_cofi_cost_func(cofi_cost_func)

All tests passed!


In [59]:
#vectorized implementation
def cofi_cost_func_v(X, W, b, Y, R, lambda_):
    ''' 
    Returns the cost for the content-based filtering
    Vectorized for speed. Uses tensorflow operations to be compatible with custom training loop

    Args:
        X (ndarray (num_movies, num_features)) : matrix of item features
        W (ndarray (num_usrs, num_features)) :matrix of user parameters
        b (ndarray (1, num_users)) : vector of user parameters
        Y (ndarray (num_movies, num_users)) : matrix of user ratings of movies
        R (ndarray (num_movies, num_users)) : matrix, where R(i,j)=1 if the i-th movie was rated
         by the j-th user
        lambda_ (float) : regularization parameter

    Returns:
        J (float) : Cost

    '''
    j = (tf.linalg.matmul(X, tf.transpose(W)) + b - Y) * R
    j = 0.5 * tf.reduce_sum(j**2) + (lambda_/2) * (tf.reduce_sum(X**2) + tf.reduce_sum(W**2))

    return J

In [60]:
#evaluate cost function
J = cofi_cost_func_v(X_r, W_r, b_r, Y_r, R_r, 0)
print(f'Cost (without regularization) : {J:0.2f}')

#evaluate cost function with regularization
print(f'Cost (with regularization) : {J:0.2f}')

Cost (without regularization) : 15.11
Cost (with regularization) : 15.11


In [65]:
#Learning Movie Recommendations
movieList, movieList_df = load_Movie_List_pd()

#initialize my ratings
my_ratings = np.zeros(num_movies)  

#check thefile small_movie_list.csv for id of each movie in our dataset
# eg - Toy Story 3 (2010) has ID 2700, so to rate it'5', you can set
my_ratings[2700] = 5

#or suppose you did not enjoy Persuasuion (2007), you can set
my_ratings[2609] = 2

#example of the following list ratings
my_ratings[929]  = 5   # Lord of the Rings: The Return of the King, The
my_ratings[246]  = 5   # Shrek (2001)
my_ratings[2716] = 3   # Inception
my_ratings[1150] = 5   # Incredibles, The (2004)
my_ratings[382]  = 2   # Amelie (Fabuleux destin d'Amélie Poulain, Le)
my_ratings[366]  = 5   # Harry Potter and the Sorcerer's Stone (a.k.a. Harry Potter and the Philosopher's Stone) (2001)
my_ratings[622]  = 5   # Harry Potter and the Chamber of Secrets (2002)
my_ratings[988]  = 3   # Eternal Sunshine of the Spotless Mind (2004)
my_ratings[2925] = 1   # Louis Theroux: Law & Disorder (2008)
my_ratings[2937] = 1   # Nothing to Declare (Rien à déclarer)
my_ratings[793]  = 5   # Pirates of the Caribbean: The Curse of the Black Pearl (2003)

my_rated = [i for i in  range(len(my_ratings)) if my_ratings[i] > 0]

print('\nNew user ratings: \n')
for i in range(len(my_ratings)):
    if my_ratings[i]>0:
        print(f'Rated {my_ratings[i]} for {movieList_df.loc[i, "title"]}')


New user ratings: 

Rated 5.0 for Shrek (2001)
Rated 5.0 for Harry Potter and the Sorcerer's Stone (a.k.a. Harry Potter and the Philosopher's Stone) (2001)
Rated 2.0 for Amelie (Fabuleux destin d'Amélie Poulain, Le) (2001)
Rated 5.0 for Harry Potter and the Chamber of Secrets (2002)
Rated 5.0 for Pirates of the Caribbean: The Curse of the Black Pearl (2003)
Rated 5.0 for Lord of the Rings: The Return of the King, The (2003)
Rated 3.0 for Eternal Sunshine of the Spotless Mind (2004)
Rated 5.0 for Incredibles, The (2004)
Rated 2.0 for Persuasion (2007)
Rated 5.0 for Toy Story 3 (2010)
Rated 3.0 for Inception (2010)
Rated 1.0 for Louis Theroux: Law & Disorder (2008)
Rated 1.0 for Nothing to Declare (Rien à déclarer) (2010)


In [79]:
#add these reviews to Y and R and normalize the ratings
#reload ratings
Y, R = load_ratings_small()

#add new user ratings to Y
Y = np.c_[my_ratings, Y]

#add new user indicator matrix to R
R = np.c_[(my_ratings != 0).astype(int), R]

#normalize the dataset
Ynorm, Ymean = normalizeRatings(Y, R)

In [80]:
#train the model using Adam optimizer
#useful values
num_movies, num_users = Y.shape
num_features = 100

#set initial parameters (W, X), use tf.Variables to track these variables
tf.random.set_seed(1234)  #for consistent results
W = tf.Variable(tf.random.normal((num_users, num_features), dtype=tf.float64), name='W')
X = tf.Variable(tf.random.normal((num_movies, num_features), dtype=tf.float64), name='X')
b = tf.Variable(tf.random.normal((1, num_users), dtype=tf.float64), name='b')

#instatiate an optimizers
optimizer = keras.optimizers.Adam(learning_rate=1e-1)

In [ ]:
#calculating derivatives using tensorflow `GradientTape()`
iterations = 200
lambda_ = 1
for iter in range(iterations):
    #use tensorflow's GradientTape() to record the operations used to cmpute the cost
    with tf.GradientTape() as tape:

        #compute the cost (forward pass included in cost)
        cost_value = cofi_cost_func_v(X, W, b, Ynorm, R, lambda_)

    #use gradient tape to automatically retrieve the gradients of 
    # the trainable variables with respect to the loss
    grads = tape.gradient( cost_value, [X,W,b] )

    #Run one step of gradient desccent by updating thevalueof the variables to minimize the loss
    optimizer.apply_gradients( zip(grads, [X,W,b] ))

    #log periodically
    if iter % 20 == 0:
        print(f'Training loss at iteration {iter}: {cost_value:0.1f}')




In [85]:
#recommendations
#make a prediction using trained weights and biases
p = np.matmul(X.numpy(), np.transpose(W.numpy())) + b.numpy()

#restore the mean
pm = p + Ymean

my_predictions = pm[:,0]

#sort predictions
ix = tf.argsort(my_predictions, direction='DESCENDING')

for i in range(17):
    j = ix[i]
    if j not in my_rated:
        print(f'Predicting rating {my_predictions[j]:0.2f} for movie {movieList[j]}')

print('\n\nOriginal vs Predicted ratings:\n')
for i in range(len(my_ratings)):
    if my_ratings[i] > 0:
        print(f'Original {my_ratings[i]}, Predicted {my_predictions[i]:0.2f} for {movieList[i]}')

Predicting rating 39.33 for movie Margin Call (2011)
Predicting rating 36.10 for movie O Brother, Where Art Thou? (2000)
Predicting rating 36.00 for movie League of Ordinary Gentlemen, A (2004)
Predicting rating 35.57 for movie Asterix and the Vikings (Astérix et les Vikings) (2006)
Predicting rating 34.47 for movie Amy's O (a.k.a. Amy's Orgasm) (2001)
Predicting rating 34.39 for movie Fuck You, Goethe (Fack Ju Göhte) (2013)
Predicting rating 33.93 for movie Giver, The (2014)
Predicting rating 33.89 for movie Louis C.K.: Oh My God (2013)
Predicting rating 33.78 for movie Powder Blue (2009)
Predicting rating 32.87 for movie Bigger, Stronger, Faster* (2008)
Predicting rating 32.85 for movie Namesake, The (2006)
Predicting rating 32.82 for movie Margaret (2011)
Predicting rating 32.60 for movie Meet the Parents (2000)
Predicting rating 32.35 for movie Divine Secrets of the Ya-Ya Sisterhood (2002)
Predicting rating 32.03 for movie Sisters (Syostry) (2001)
Predicting rating 31.64 for movie 

In [88]:
#using pandas data frame
filter = (movieList_df['number of ratings'] > 20)
movieList_df['pred'] = my_predictions
movieList_df = movieList_df.reindex(columns=['pred', 'mean rating', 'number of ratings', 'title'])
movieList_df.loc[ix[:300]].loc[filter].sort_values('mean rating', ascending=False)

,pred,mean rating,number of ratings,title
848,20.682134,4.033784,74,Lost in Translation (2003)
395,27.708253,4.000000,123,"Beautiful Mind, A (2001)"
3862,27.246243,3.986111,36,Kingsman: The Secret Service (2015)
2756,22.395304,3.954545,22,"Town, The (2010)"
451,22.059193,3.952381,21,And Your Mother Too (Y tu mamá también) (2001)
642,19.956191,3.945652,46,Adaptation (2002)
2314,25.169183,3.945652,46,Gran Torino (2008)
916,22.666512,3.933333,30,Battle Royale (Batoru rowaiaru) (2000)
3556,22.733851,3.916667,54,"Wolf of Wall Street, The (2013)"
3665,22.570805,3.833333,30,X-Men: Days of Future Past (2014)
